# Tech Challenge - Fase 1 | Hospital Universitário: IA para Diagnóstico de Câncer de Mama

**Curso:** Pós-Tech em Machine Learning / IA
**Projeto:** Sistema Inteligente de Suporte ao Diagnóstico Médico
**Dataset:** Diagnóstico de Câncer de Mama (`data.csv` — Breast Cancer Wisconsin)

**Integrantes:**

- Bianca Maciel - RM376200
- Eduardo O Charrone - RM377641
- Guilherme Mendes Alburquerque - RM378723
- Igor Matos de Andrade - RM377344
- João Augusto Gonçalves de Aragão - RM377385

---
### Contexto e Objetivo
Sistema de IA (Machine Learning) para auxiliar médicos na triagem de exames de câncer de mama
(**Maligno** vs. **Benigno**). A métrica prioritária é o **recall da classe maligna** (sensibilidade):
deixar de identificar um tumor maligno (falso negativo) tem custo clínico muito maior que um falso
positivo. O modelo é ferramenta de **suporte** — a palavra final é sempre do médico.

## 1. Configuração de ambiente e dependências

In [ ]:
import sys
import os
import subprocess
from pathlib import Path

# Portabilidade: roda igual em Local (VSCode/Jupyter), Google Colab e Docker.
EM_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/iigormatos/tech-challenge-fase1-v1.git"

if EM_COLAB:
    destino = Path("/content/tech-challenge-fase1-v1")
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
    os.chdir(destino)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    raiz = destino
else:
    # Funciona abrindo de notebooks/ (local) ou da raiz (Docker/nbconvert).
    raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

# Caminhos e semente vêm SEMPRE de src/config.py (sem valores hardcoded).
from src import data as data_mod
from src.config import RANDOM_STATE, CANCER_MAMA_CSV, PROJECT_ROOT

print("Ambiente:", "Colab" if EM_COLAB else "Local/Docker")
print("Raiz do projeto:", PROJECT_ROOT)
print("Semente (RANDOM_STATE):", RANDOM_STATE)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import shap
import joblib

from statsmodels.stats.proportion import proportion_confint

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    recall_score, precision_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve,
    precision_recall_curve, ConfusionMatrixDisplay,
)

In [ ]:
# Estética global e helper único para salvar figuras em reports/figures.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

DIR_FIGURAS = PROJECT_ROOT / "reports" / "figures"
DIR_MODELOS = PROJECT_ROOT / "models"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
DIR_MODELOS.mkdir(parents=True, exist_ok=True)

def salvar_figura(fig, nome):
    """Salva a figura em reports/figures/<nome>.png e retorna o caminho."""
    caminho = DIR_FIGURAS / f"{nome}.png"
    fig.savefig(caminho, dpi=120, bbox_inches="tight")
    return caminho

## 2. Carregamento e qualidade dos dados

In [ ]:
# Carga centralizada (src/data.py): descarta id/colunas vazias, traduz colunas 1x, codifica alvo.
df = data_mod.carregar_dados()
print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas (30 features + alvo)")

qualidade = data_mod.checar_qualidade(df)
print("Total de nulos:", qualidade["total_nulos"])
print("Duplicatas:", qualidade["duplicatas"])
print("\nDistribuição de classes (0=benigno, 1=maligno):")
print(qualidade["distribuicao_classes"].to_string())
print("\nProporção (%):")
print(qualidade["distribuicao_classes_pct"].to_string())

df.describe()

**Interpretação.** A base tem 569 pacientes e 30 features numéricas, sem valores nulos e sem
duplicatas — a coluna `id` (sem valor estatístico) e a coluna espúria `Unnamed: 32` (vazia) já são
descartadas na carga. A distribuição é **desbalanceada (~63% benigno / ~37% maligno)**, o que torna a
**acurácia** uma métrica enganosa: um classificador que chutasse sempre "benigno" acertaria 63%.
Tratamos o desbalanceamento com ponderação de classes (seção 5), não com reamostragem, preservando a
distribuição real nos três conjuntos.

In [ ]:
# Checagem visual de ausentes (missingno). A base não tem nulos, confirmado graficamente.
ax = msno.matrix(df, fontsize=9)
salvar_figura(ax.get_figure(), "01_matriz_ausentes")
plt.show()

## 3. Análise exploratória (EDA) por classe

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
cont = sns.countplot(x="diagnostico", data=df, hue="diagnostico", palette="Set1", legend=False, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Benigno (0)", "Maligno (1)"])
ax.set_title("Distribuição de diagnóstico"); ax.set_xlabel("Diagnóstico"); ax.set_ylabel("Pacientes")
for p in cont.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="center", xytext=(0, 8), textcoords="offset points")
salvar_figura(fig, "02_distribuicao_classes"); plt.show()

**Interpretação.** Confirma o desbalanceamento 357 × 212. Por isso a avaliação (seção 7) prioriza
recall maligno, especificidade e PR-AUC em vez de acurácia, e os modelos usam `class_weight="balanced"`.

In [ ]:
# Boxplots por classe: o gráfico mais informativo nesta base.
features_destaque = [
    "raio_medio", "textura_media", "perimetro_medio", "area_media",
    "concavidade_media", "pontos_concavos_medio", "perimetro_pior", "pontos_concavos_pior",
]
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, feat in zip(axes.ravel(), features_destaque):
    sns.boxplot(x="diagnostico", y=feat, data=df, hue="diagnostico", palette="Set2", legend=False, ax=ax)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Benigno", "Maligno"]); ax.set_xlabel(""); ax.set_title(feat)
fig.suptitle("Distribuição das features por classe (benigno x maligno)", y=1.02)
fig.tight_layout(); salvar_figura(fig, "03_boxplots_por_classe"); plt.show()

**Interpretação.** Tumores malignos têm, de forma consistente, valores **maiores** de raio,
perímetro, área, concavidade e pontos côncavos. As duas classes se separam bem justamente nas features
de tamanho/forma — o que explica o alto desempenho dos modelos. Note que os malignos apresentam uma
cauda de valores altos: são tumores grandes reais, não ruído (quantificado a seguir).

In [ ]:
# Quantifica o impacto da (antiga) remoção de outliers por IQR, por classe.
features = df.drop(columns=["diagnostico"]).columns
Q1 = df[features].quantile(0.25); Q3 = df[features].quantile(0.75); IQR = Q3 - Q1
fora = ((df[features] < (Q1 - 1.5 * IQR)) | (df[features] > (Q3 + 1.5 * IQR))).any(axis=1)

tab = (df.assign(fora_iqr=fora).groupby("diagnostico")["fora_iqr"].mean() * 100).round(1)
tab.index = ["Benigno", "Maligno"]
print("% de amostras com >=1 feature fora do IQR, por classe:")
print(tab.to_string())

malignos_fora = int(fora[df["diagnostico"] == 1].sum())
malignos_total = int((df["diagnostico"] == 1).sum())
print(f"\nRemover linhas fora do IQR descartaria {malignos_fora} de {malignos_total} malignos "
      f"({malignos_fora / malignos_total * 100:.0f}%).")

**Interpretação (correção metodológica).** A versão anterior do projeto removia "outliers" por IQR
**antes** do split. Como se vê, a maioria dos casos **malignos** cai fora do IQR, contra uma fração
pequena dos benignos: remover essas linhas descartaria mais da metade dos malignos e tornaria o problema
artificialmente fácil, além de enviesar o conjunto de teste. **Decisão: não remover nenhuma linha.** Os
extremos são sinal clínico (tumores grandes), não ruído. Para os modelos sensíveis a escala usamos
`StandardScaler` dentro do pipeline (ajustado só no treino); árvores não precisam de escala.

In [ ]:
# Correlação de CADA feature com o alvo (base completa) + multicolinearidade.
corr = df.corr(numeric_only=True)
corr_alvo = corr["diagnostico"].drop("diagnostico").sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
corr_alvo.plot(kind="barh", color=np.where(corr_alvo > 0, "#c0392b", "#2980b9"), ax=ax)
ax.set_title("Correlação de cada feature com o diagnóstico (maligno=1)")
ax.set_xlabel("Correlação de Pearson")
salvar_figura(fig, "04_correlacao_alvo"); plt.show()

grupo_media = [c for c in df.columns if c.endswith("_media") or c.endswith("_medio")]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[grupo_media].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Multicolinearidade entre features (grupo média)")
salvar_figura(fig, "05_heatmap_multicolinearidade"); plt.show()

print("raio x perímetro:", round(df["raio_medio"].corr(df["perimetro_medio"]), 3))
print("raio x área:", round(df["raio_medio"].corr(df["area_media"]), 3))

**Interpretação.** As features de tamanho (raio, perímetro, área, concavidade, pontos côncavos)
têm correlação positiva forte com malignidade. Há **multicolinearidade** alta: raio × perímetro ≈ 0,998
e raio × área ≈ 0,987 — são praticamente a mesma informação geométrica. Isso importa na leitura dos
coeficientes da Regressão Logística e do SHAP (seção 8): a importância de um grupo colinear tende a se
distribuir/instabilizar entre suas features, então não se deve ler o coeficiente de uma única delas de
forma ingênua.

## 4. Pré-processamento e divisão (sem vazamento)

In [ ]:
# Split estratificado 70/15/15 (src/data.py), ANTES de qualquer transformação dependente dos dados.
X_treino, X_val, X_teste, y_treino, y_val, y_teste = data_mod.dividir_treino_val_teste(df)
for nome, y in [("Treino", y_treino), ("Validação", y_val), ("Teste", y_teste)]:
    print(f"{nome:10s}: {len(y):3d} amostras | malignos {y.mean() * 100:.1f}%")

def criar_pipeline(modelo, escalar):
    """Pipeline imputer -> [scaler] -> modelo. Transformações ajustadas SÓ no treino/fold."""
    etapas = [("imputer", SimpleImputer(strategy="median"))]
    if escalar:
        etapas.append(("scaler", StandardScaler()))
    etapas.append(("modelo", modelo))
    return Pipeline(etapas)

**Papéis dos conjuntos.** **Treino** (70%) ajusta transformações e modelos; **Validação** (15%)
serve para tuning e escolha do limiar de decisão; **Teste** (15%) é tocado uma única vez, ao final.
Toda transformação dependente dos dados (imputação da mediana, escala) vive **dentro** do `Pipeline`, de
modo que na validação cruzada cada fold ajusta o scaler apenas com seus próprios dados de treino — sem o
vazamento que existia antes (scaler ajustado sobre todos os dados).

## 5. Modelagem: catálogo, tuning e validação cruzada

In [ ]:
def catalogo_modelos(random_state=RANDOM_STATE):
    """4 modelos (edital exige >=2). LR/RF/SVM com class_weight; KNN sem (depende só do limiar)."""
    return [
        {"nome": "Regressão Logística",
         "estimador": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=random_state),
         "escalar": True, "ponderado": True, "grade": {"modelo__C": [0.01, 0.1, 1, 10]}},
        {"nome": "Random Forest",
         "estimador": RandomForestClassifier(class_weight="balanced", random_state=random_state),
         "escalar": False, "ponderado": True,
         "grade": {"modelo__n_estimators": [200, 400], "modelo__max_depth": [None, 5, 10]}},
        {"nome": "SVM (RBF)",
         "estimador": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=random_state),
         "escalar": True, "ponderado": True,
         "grade": {"modelo__C": [0.1, 1, 10], "modelo__gamma": ["scale", "auto"]}},
        {"nome": "KNN",
         "estimador": KNeighborsClassifier(),
         "escalar": True, "ponderado": False, "grade": {"modelo__n_neighbors": [3, 5, 7, 9, 11]}},
    ]

def tunar(item, X, y, cv_splits=5):
    """GridSearchCV (scoring=recall) com StratifiedKFold, SÓ sobre o treino."""
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    pipe = criar_pipeline(item["estimador"], item["escalar"])
    busca = GridSearchCV(pipe, item["grade"], scoring="recall", cv=cv, n_jobs=-1)
    busca.fit(X, y)
    return busca.best_estimator_, busca.best_params_

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
modelos = catalogo_modelos()
ajustados = {}
cv_resultados = {}

for item in modelos:
    pipe_tunado, best = tunar(item, X_treino, y_treino)
    ajustados[item["nome"]] = {"pipeline": pipe_tunado, "ponderado": item["ponderado"]}
    scores = cross_validate(pipe_tunado, X_treino, y_treino, cv=cv, scoring=["recall", "roc_auc"])
    cv_resultados[item["nome"]] = scores
    print(f'{item["nome"]:22s} | melhores params: {best}')
    print(f'{"":22s} | CV recall {scores["test_recall"].mean():.3f} +/- {scores["test_recall"].std():.3f}'
          f' | CV ROC-AUC {scores["test_roc_auc"].mean():.3f}')

**Interpretação.** Os hiperparâmetros são ajustados por validação cruzada estratificada **no
treino**, otimizando recall. O recall de CV é reportado como **média ± desvio** — importante porque
diferenças pequenas entre modelos podem estar dentro do ruído e não sustentam, sozinhas, a escolha do
"melhor" modelo.

## 6. Limiar de decisão (escolhido na validação)

In [ ]:
def escolher_limiar(pipeline, X_v, y_v, piso_especificidade=0.90):
    """Maximiza recall maligno sujeito a especificidade >= piso. SÓ na validação.

    Evita a solução degenerada (classificar tudo como maligno -> recall 100%, especificidade 0).
    """
    y_v = np.asarray(y_v)
    probs = pipeline.predict_proba(X_v)[:, 1]
    melhor_lim, melhor_rec = 0.5, -1.0
    for lim in np.linspace(0.01, 0.99, 99):
        pred = (probs >= lim).astype(int)
        tn = int(((pred == 0) & (y_v == 0)).sum()); fp = int(((pred == 1) & (y_v == 0)).sum())
        tp = int(((pred == 1) & (y_v == 1)).sum()); pos = int((y_v == 1).sum())
        espec = tn / (tn + fp) if (tn + fp) else 0.0
        rec = tp / pos if pos else 0.0
        if espec >= piso_especificidade and rec > melhor_rec:
            melhor_rec, melhor_lim = rec, float(lim)
    return melhor_lim

PISO_ESPECIFICIDADE = 0.90
limiares = {nome: escolher_limiar(d["pipeline"], X_val, y_val, PISO_ESPECIFICIDADE)
            for nome, d in ajustados.items()}
for nome, l in limiares.items():
    print(f"{nome:22s} | limiar = {l:.2f}")

**Interpretação.** O limiar não é fixado em 0,5: para cada modelo escolhemos, **na validação**, o
corte que maximiza o recall maligno mantendo a especificidade ≥ 0,90. Assim priorizamos a sensibilidade
(poucos falsos negativos) sem liberar falsos positivos demais. O limiar é então aplicado **inalterado**
ao teste.

## 7. Avaliação final (teste) e comparação

In [ ]:
def ic_recall_wilson(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    tp = int(((y_pred == 1) & (y_true == 1)).sum()); n = int((y_true == 1).sum())
    low, high = proportion_confint(tp, n, method="wilson")
    return low, high

def avaliar(pipeline, X, y, limiar):
    """Métricas no ponto de operação definido por `limiar`."""
    y = np.asarray(y)
    probs = pipeline.predict_proba(X)[:, 1]
    pred = (probs >= limiar).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    ic_low, ic_high = ic_recall_wilson(y, pred)
    return {
        "recall": recall_score(y, pred, zero_division=0), "recall_ic": (ic_low, ic_high),
        "precision": precision_score(y, pred, zero_division=0), "f1": f1_score(y, pred, zero_division=0),
        "especificidade": tn / (tn + fp) if (tn + fp) else 0.0,
        "roc_auc": roc_auc_score(y, probs), "pr_auc": average_precision_score(y, probs),
        "falsos_negativos": int(fn),
    }

def linha(res, modelo, ponderado, conjunto):
    return {
        "Modelo": modelo, "Ponderado": "sim" if ponderado else "não (só limiar)", "Conjunto": conjunto,
        "Recall maligno": round(res["recall"], 3),
        "IC recall (Wilson)": f'[{res["recall_ic"][0]:.2f}, {res["recall_ic"][1]:.2f}]',
        "Precision": round(res["precision"], 3), "F1": round(res["f1"], 3),
        "Especificidade": round(res["especificidade"], 3),
        "ROC-AUC": round(res["roc_auc"], 3), "PR-AUC": round(res["pr_auc"], 3),
        "FN": res["falsos_negativos"],
    }

linhas, resultados_teste = [], {}
for nome, d in ajustados.items():
    l = limiares[nome]
    r_val = avaliar(d["pipeline"], X_val, y_val, l)
    r_te = avaliar(d["pipeline"], X_teste, y_teste, l)
    resultados_teste[nome] = r_te
    linhas.append(linha(r_val, nome, d["ponderado"], "Validação"))
    linhas.append(linha(r_te, nome, d["ponderado"], "Teste"))

tabela = pd.DataFrame(linhas)
tabela.to_csv(PROJECT_ROOT / "reports" / "tabela_comparativa.csv", index=False)
tabela

In [ ]:
# Curvas ROC, Precision-Recall e recall x limiar (teste).
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for nome, d in ajustados.items():
    probs = d["pipeline"].predict_proba(X_teste)[:, 1]
    fpr, tpr, _ = roc_curve(y_teste, probs); axes[0].plot(fpr, tpr, label=nome)
    prec, rec, _ = precision_recall_curve(y_teste, probs); axes[1].plot(rec, prec, label=nome)
    lims = np.linspace(0.01, 0.99, 99)
    yv = np.asarray(y_teste)
    recs = [((probs >= t).astype(int)[yv == 1]).mean() for t in lims]
    axes[2].plot(lims, recs, label=nome)
axes[0].plot([0, 1], [0, 1], "k--"); axes[0].set_title("Curva ROC (teste)")
axes[0].set_xlabel("1 - especificidade (FPR)"); axes[0].set_ylabel("Recall (TPR)"); axes[0].legend()
axes[1].set_title("Curva Precision-Recall (teste)"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend()
axes[2].set_title("Recall maligno x limiar (teste)"); axes[2].set_xlabel("Limiar"); axes[2].set_ylabel("Recall maligno"); axes[2].legend()
fig.tight_layout(); salvar_figura(fig, "06_curvas"); plt.show()

In [ ]:
# Matrizes de confusão no teste (com o limiar de cada modelo).
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, (nome, d) in zip(axes, ajustados.items()):
    probs = d["pipeline"].predict_proba(X_teste)[:, 1]
    pred = (probs >= limiares[nome]).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_teste, pred, display_labels=["Benigno", "Maligno"],
                                            cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"{nome}\n(limiar {limiares[nome]:.2f})")
fig.tight_layout(); salvar_figura(fig, "07_matrizes_confusao"); plt.show()

**Interpretação.** A tabela mostra as seis métricas em validação e teste, lado a lado, para os
quatro modelos. O foco é o **recall maligno** e a contagem de **falsos negativos (FN)** — cada FN é um
câncer não sinalizado. Como o conjunto de teste é pequeno, reportamos o **intervalo de confiança de
Wilson** do recall: só faz sentido afirmar que um modelo é superior a outro quando seus intervalos **não
se sobrepõem**. ROC-AUC/PR-AUC resumem o desempenho independente do limiar; a curva recall × limiar
mostra o trade-off clínico de mover o ponto de operação.

## 8. Interpretabilidade

In [ ]:
nomes_features = list(X_treino.columns)
pipe_rf = ajustados["Random Forest"]["pipeline"]
pipe_lr = ajustados["Regressão Logística"]["pipeline"]

# SHAP global — Random Forest (TreeExplainer). Usa dados após imputação (RF não escala).
modelo_rf = pipe_rf.named_steps["modelo"]
X_teste_rf = pipe_rf[:-1].transform(X_teste)
expl_rf = shap.TreeExplainer(modelo_rf)
sv_rf = expl_rf.shap_values(X_teste_rf)
if isinstance(sv_rf, list):
    sv_rf_pos = sv_rf[1]
elif getattr(sv_rf, "ndim", 2) == 3:
    sv_rf_pos = sv_rf[:, :, 1]
else:
    sv_rf_pos = sv_rf

shap.summary_plot(sv_rf_pos, X_teste_rf, feature_names=nomes_features, show=False)
salvar_figura(plt.gcf(), "08_shap_rf_global"); plt.show()

In [ ]:
# Coeficientes padronizados da Regressão Logística (modelo mais interpretável).
coefs = pd.Series(pipe_lr.named_steps["modelo"].coef_[0], index=nomes_features).sort_values()
fig, ax = plt.subplots(figsize=(8, 10))
coefs.plot(kind="barh", color=np.where(coefs > 0, "#c0392b", "#2980b9"), ax=ax)
ax.set_title("Coeficientes padronizados - Regressão Logística (positivo -> maligno)")
salvar_figura(fig, "09_coeficientes_lr"); plt.show()

In [ ]:
# SHAP individual: 1 caso maligno e 1 benigno (sustenta o uso clínico, caso a caso).
base = expl_rf.expected_value
base_pos = base[1] if isinstance(base, (list, np.ndarray)) else base
expl = shap.Explanation(values=sv_rf_pos, base_values=base_pos,
                        data=np.asarray(X_teste_rf), feature_names=nomes_features)
yv = np.asarray(y_teste)
idx_mal = int(np.where(yv == 1)[0][0])
idx_ben = int(np.where(yv == 0)[0][0])
for idx, rot in [(idx_mal, "maligno"), (idx_ben, "benigno")]:
    shap.plots.waterfall(expl[idx], show=False)
    salvar_figura(plt.gcf(), f"10_shap_individual_{rot}"); plt.show()

In [ ]:
# Permutation importance no TESTE (complementa a importância por Gini, que é enviesada).
perm = permutation_importance(pipe_rf, X_teste, y_teste, scoring="recall",
                              n_repeats=20, random_state=RANDOM_STATE)
imp = pd.Series(perm.importances_mean, index=nomes_features).sort_values()
fig, ax = plt.subplots(figsize=(8, 10))
imp.tail(15).plot(kind="barh", color="teal", ax=ax)
ax.set_title("Permutation importance no teste (recall) - Random Forest")
salvar_figura(fig, "11_permutation_rf"); plt.show()

**Interpretação.** As três visões convergem: features de **tamanho/forma do pior núcleo**
(`pontos_concavos_pior`, `perimetro_pior`, `raio_pior`, `area_pior`, `concavidade_media`) são as que mais
empurram a predição para maligno — coerente com a biologia (tumores malignos são maiores e de contorno
mais irregular). Os coeficientes da Regressão Logística apontam na mesma direção. **Ressalva de
multicolinearidade:** como raio, perímetro e área são quase redundantes, a importância se distribui entre
elas; a leitura correta é "o grupo de tamanho importa", não uma feature isolada. O `permutation_importance`
no teste confirma as mesmas features sem o viés da importância de Gini.

## 9. Persistência do modelo

In [ ]:
# Modelo final = maior recall na validação; persistido com seu limiar para a demonstração.
melhor_nome = max(ajustados, key=lambda n: avaliar(ajustados[n]["pipeline"], X_val, y_val, limiares[n])["recall"])
pipe_final = ajustados[melhor_nome]["pipeline"]
limiar_final = limiares[melhor_nome]

joblib.dump({"pipeline": pipe_final, "limiar": limiar_final, "features": nomes_features},
            DIR_MODELOS / "modelo_final.joblib")
with open(DIR_MODELOS / "limiar.json", "w", encoding="utf-8") as fp:
    json.dump({"modelo": melhor_nome, "limiar": round(limiar_final, 3),
               "piso_especificidade": PISO_ESPECIFICIDADE}, fp, indent=2, ensure_ascii=False)

print("Modelo final:", melhor_nome, "| limiar:", round(limiar_final, 3))
print("Salvo em:", DIR_MODELOS / "modelo_final.joblib")

## 10. Conclusão crítica

### Escolha da métrica
Em rastreamento de câncer, o erro mais grave é o **falso negativo** (liberar um paciente com tumor
maligno). Por isso a métrica prioritária é o **recall da classe maligna**, e não a **acurácia** — que,
numa base 63/37, é inflada pela classe benigna e esconde justamente os erros que mais custam. Reportamos
também precision, F1, **especificidade**, ROC-AUC e PR-AUC para dar o quadro completo do trade-off entre
sensibilidade (recall) e falsos positivos.

### Comparação dos modelos
Os quatro modelos (Regressão Logística, Random Forest, SVM-RBF e KNN) são comparados na tabela da seção
7, com o limiar de cada um escolhido na validação (especificidade ≥ 0,90). A decisão sobre "qual é
melhor" respeita os **intervalos de confiança** do recall: com um conjunto de teste pequeno, diferenças
de uma ou duas amostras não são conclusivas. Os falsos negativos de cada modelo estão nas matrizes de
confusão e na coluna FN da tabela.

### Uso prático no hospital
O sistema atua como **módulo de triagem**: casos sinalizados como malignos sobem na fila de análise da
radiologia/oncologia. O **SHAP** e os coeficientes dão transparência (por que o modelo alertou),
apoiando a auditoria clínica. **Soberania médica:** o diagnóstico e o laudo são de responsabilidade
exclusiva do médico; a IA é suporte à decisão.

### Limitações
- Base pequena (569 pacientes) e de **centro único** (Wisconsin), coletada em **1995** — pode não
  generalizar para outras populações/equipamentos.
- Conjunto de teste pequeno → métricas com incerteza (daí os intervalos de confiança).
- Features derivadas de imagem já processada; não cobre o pipeline de aquisição real.

### Para uso em produção faltaria
Validação externa (outros centros), **calibração de probabilidades**, monitoramento de *data drift*,
e um protocolo claro de encaminhamento dos casos sinalizados. A remoção de outliers da versão anterior
foi **corrigida**: descartava mais da metade dos malignos e tornava o resultado otimista demais.